# Z-PEFT-style detector on the shared development split (Sandra)

CPU runtime. Validate the custodian's development bundle, fit with the existing grouped CV on train, freeze, then calibrate on independent benign records. Evaluation and final assessment remain with the custodian.

In [ ]:
!pip -q install safetensors scikit-learn
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = "https://github.com/JuliaIssa2/FYP-AI-Antivirus.git"
SOURCE_COMMIT = ""  # full reviewed 40-character commit supplied by the custodian
BUNDLE = "/content/drive/MyDrive/fyp_shared/development.json"
BUNDLE_SHA256 = ""   # trusted file checksum, supplied independently
BINDING_SHA256 = ""  # trusted ProtocolBinding fingerprint
ADAPTER_ROOT = "/content/adapters"            # <root>/<adapter_id>/adapter_model.safetensors (same staging as the PEFTGuard notebook)
CACHE_DIR  = "/content/zpeft_cache"           # feature cache keyed by file SHA256 + extractor version
RUN_DIR    = "/content/runs/zpeft_seed0"
DRIVE_OUT  = "/content/drive/MyDrive/fyp_shared/zpeft_seed0"
SEED = 0

In [ ]:
import os, sys, shutil, re, subprocess
from pathlib import Path

if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise ValueError("Supply the full reviewed source commit from the custodian")
repo = Path("/content/repo")
if not repo.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(repo)], check=True)
elif subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Use a clean reviewed checkout")
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", SOURCE_COMMIT], check=True)
if subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError("Source commit mismatch")
if any(k == "shared_protocol" or k.startswith("shared_protocol.") for k in sys.modules):
    raise RuntimeError("Restart the runtime before loading the reviewed source")
os.chdir(repo)
sys.path.insert(0, str(repo))
print("Reviewed source:", SOURCE_COMMIT)

from shared_protocol.development import load_development_bundle

development = load_development_bundle(
    BUNDLE, expected_file_sha256=BUNDLE_SHA256, expected_binding_sha256=BINDING_SHA256)
binding = development.binding
train_records = development.records("train")
val_records = development.records("validation")
cal_records = development.records("calibration")
print({p: len(development.records(p)) for p in ("train", "validation", "calibration")})

Verify custodian-staged neutral-ID files. Existing files are re-hashed, never silently skipped. No adapter downloads or Hugging Face credentials are used.

In [ ]:
from shared_protocol.development import verify_development_checkpoints

# Custodian-staged files only. Existing files are always hashed; no downloads.
resolver = verify_development_checkpoints(
    development, ADAPTER_ROOT, expected_binding_sha256=BINDING_SHA256)
print("Verified all staged development checkpoints")

## Fit (train split only; every file is re-hashed against the manifest first)

In [ ]:
from shared_protocol.detectors.zpeft.train import ZPeftConfig, fit
from shared_protocol.detectors.peftguard.converter import root_resolver
cfg = ZPeftConfig(seed=SEED)
result = fit(binding, train_records, val_records, cfg, RUN_DIR, resolver=root_resolver(ADAPTER_ROOT, restricted=True), cache_dir=CACHE_DIR)
print(result["model_file_sha256"], "<- record this in the team log (the frozen state)")
os.makedirs(DRIVE_OUT, exist_ok=False)
shutil.copyfile(result["model_file"], f"{DRIVE_OUT}/model.json")

## Freeze: clean-only 5% conformal threshold (calibration partition only)

In [ ]:
from shared_protocol.detectors.zpeft.detector import ZPeftDetector
from shared_protocol.scoring import score_development
from shared_protocol.calibration import calibrate_benign, save_threshold
detector = ZPeftDetector.from_model(result["model_file"], binding, expected_file_sha256=result["model_file_sha256"],
                                    resolver=root_resolver(ADAPTER_ROOT, restricted=True), cache_dir=CACHE_DIR)
cal_scores = score_development(detector, cal_records, binding, "calibration")
threshold = calibrate_benign(binding, cal_records, cal_scores)
print("n calibration:", len(cal_scores.scores), "| threshold:", threshold.value)
save_threshold(f"{DRIVE_OUT}/threshold.json", threshold, binding)

## Custodian evaluation only

Send the frozen model, threshold, hashes, source commit and run configuration to the custodian. This development notebook contains no evaluation or final-holdout loader; the separate custodian workflow handles assessment after freeze.